In [ ]:
# ============================================================================
# CELL 1: SETUP & IMPORTS - PRODUCTION READY (UNBUFFERED LOGS)
# ============================================================================
import subprocess
import sys
import os
import random
import warnings
import math
import json
import csv
import time
import logging
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR, OneCycleLR

# --- FIX: Force Unbuffered Output (Crucial for debugging crashes) ---
os.environ['PYTHONUNBUFFERED'] = '1'

# --- AMP utilities ---
if int(torch.__version__.split('.')[0]) >= 2:
    from torch.amp import GradScaler, autocast
else:
    from torch.cuda.amp import GradScaler, autocast

# --- Optional libraries with fallbacks ---
# Gom nhóm import để code gọn hơn
try:
    import cv2
    from PIL import Image
    import matplotlib.pyplot as plt
    import seaborn as sns
    from tqdm.notebook import tqdm
except ImportError as e:
    print(f"[WARN] Optional library missing: {e}. Visualization/Processing may be limited.")
    # Fallback cho tqdm nếu không chạy trong notebook
    try:
        from tqdm import tqdm
    except ImportError:
        tqdm = None

# --- TorchVision & COCO utils ---
try:
    import torchvision
    from torchvision.models.detection import RetinaNet
    from torchvision.models.detection.retinanet import RetinaNetClassificationHead
except ImportError:
    print("[WARN] torchvision not available. Model training will fail.")

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except ImportError:
    COCO = None
    COCOeval = None
    print("[WARN] pycocotools not available. COCO evaluation will fail.")

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ImportError:
    A = None
    print("[WARN] albumentations not available. Data augmentation will fail.")

# Fix multiprocessing sharing strategy
try:
    torch.multiprocessing.set_sharing_strategy('file_system')
except RuntimeError:
    pass

# ----------------------------------------------------------------------------
# Paths & Logger setup
# ----------------------------------------------------------------------------
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

class ForceFlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

# Logger setup
LOG_PATH = WORK_DIR / "training.log"
root_logger = logging.getLogger()
for h in list(root_logger.handlers):
    root_logger.removeHandler(h)

file_handler = logging.FileHandler(str(LOG_PATH))
stream_handler = ForceFlushHandler(sys.stdout) 

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] [%(levelname)s] %(message)s',
    handlers=[file_handler, stream_handler],
    force=True
)
logger = logging.getLogger(__name__)
logger.propagate = False

# ----------------------------------------------------------------------------
# Environment & Reproducibility
# ----------------------------------------------------------------------------
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='torch.nn.parallel')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'max_split_size_mb:128')
os.environ.setdefault('CUDA_LAUNCH_BLOCKING', '0')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

set_seed(42, deterministic=False)

# ----------------------------------------------------------------------------
# TF32 optimization
# ----------------------------------------------------------------------------
if torch.cuda.is_available():
    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = True

# ----------------------------------------------------------------------------
# Runtime info
# ----------------------------------------------------------------------------
logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
logger.info(f"Device set to: {device}")
logger.info("Setup Complete. Ready for MobileNetV3 optimization.")

In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION - OPTIMIZED FOR MOBILENETV3 + EMA
# ============================================================================
from typing import Dict, List
from pathlib import Path
import os

# Define Class Names explicitly to ensure consistency across Train/Val/Test
NUM_USER_CLASSES = 11
USER_CLASSES_LIST = [
    'Vehicle', 'Bus', 'Bicycle', 'Person', 'Engine', 'Truck',
    'Tricycle', 'Obstacle', 'Pothole', 'Traffic Light', 'Traffic Sign'
]

CONFIG: Dict = {
    # --- DATASET PATHS ---
    'TRAIN_ANN': Path('/kaggle/input/traffic-object/coco_split/coco_split/annotations/instances_train.json'),
    'TRAIN_IMG': Path('/kaggle/input/traffic-object/coco_split/coco_split/train'),
    'VAL_ANN': Path('/kaggle/input/traffic-object/coco_split/coco_split/annotations/instances_val.json'),
    'VAL_IMG': Path('/kaggle/input/traffic-object/coco_split/coco_split/val'),
    'TEST_ANN': Path('/kaggle/input/traffic-object/coco_split/coco_split/annotations/instances_test.json'),
    'TEST_IMG': Path('/kaggle/input/traffic-object/coco_split/coco_split/test'),
    
    # --- OUTPUT ARTIFACTS ---
    'OUTPUT_DIR': Path('/kaggle/working'),
    'CHECKPOINT_DIR': Path('/kaggle/working/checkpoints'),
    
    # --- TRAINING HYPERPARAMETERS (MobileNetV3 Optimized) ---
    'EPOCHS': 30,
    'BATCH_SIZE': 16,        # 32 is safe for MobileNetV3 on P100/T4
    'ACCUMULATION_STEPS': 2, # Batch 32 is sufficient for BN stability
    'NUM_WORKERS': 2,
    'PIN_MEMORY': True,
    'PERSISTENT_WORKERS': True,
    'PREFETCH_FACTOR': 2,
    
    # --- OPTIMIZER & SCHEDULER ---
    'OPTIMIZER': 'AdamW',      
    'LEARNING_RATE': 5e-5,   # Higher start LR for lightweight backbone
    'WEIGHT_DECAY': 1e-4,
    'WARMUP_EPOCHS': 1,
    'GRAD_CLIP': 1.0,        # Relaxed clipping for faster convergence
    'USE_AMP': True,         # Automatic Mixed Precision
    
    # --- LOSS FUNCTION ---
    'FOCAL_ALPHA': 0.25,
    'FOCAL_GAMMA': 2.0,
    'CIOU_SCALE': 2.0,
    # --- MULTI-SCALE TRAINING ---
    'MULTI_SCALE': True,
    'INPUT_SIZES': [416, 448, 480], # Sweet spot for MobileNetV3 speed/acc
    'SCALE_CHANGE_FREQUENCY': 200,  # Change scale every 200 batches
    'BASE_INPUT_SIZE': 448,
    
    # --- EMA CONFIGURATION (NEW) ---
    'EMA_ENABLED': True,     # Master switch for EMA
    'EMA_DECAY': 0.9999,     # 0.9999 for >20 epochs, 0.9995 for short runs
    
    # --- DATA FILTERING ---
    'MIN_BOX_SIZE': 4.0,     
    'MIN_BOXES_PER_IMAGE': 1,
    
    # --- LOGGING & EVALUATION ---
    'VAL_FREQUENCY': 2,      
    'SCORE_THRESH': 0.05,
    'EARLY_STOPPING_PATIENCE': 8,
    'VAL_SUBSET': 1200,      # Limit validation size to speed up epoch
    'LOG_BATCH_INTERVAL': 50,
    'LOG_LEVEL': 'INFO',
    'BEST_METRIC': 'mAP',
    'SEED': 42,
    
    # --- RESUME & EXPORT ---
    'RESUME_TRAINING': False, 
    'RESUME_CHECKPOINT': None, 
    'PRETRAINED_WEIGHTS': ('/kaggle/input/checkpoint-mobilenetv3-2/checkpoints/best.pth'),
    'RESUME_LOGS_DIR': None,   
    'START_EPOCH': 0,
    'RUN_TEST_EVAL': True,
    'EXPORT_MODEL': True,
}

# Derived config
CONFIG['NUM_MODEL_CLASSES'] = len(USER_CLASSES_LIST) + 1 # +1 for background

# Ensure directories exist
CONFIG['OUTPUT_DIR'].mkdir(parents=True, exist_ok=True)
CONFIG['CHECKPOINT_DIR'].mkdir(parents=True, exist_ok=True)
(CONFIG['OUTPUT_DIR'] / 'logs').mkdir(parents=True, exist_ok=True)
(CONFIG['OUTPUT_DIR'] / 'plots').mkdir(parents=True, exist_ok=True)

logger.info(f"Config Loaded. Batch: {CONFIG['BATCH_SIZE']} | Backbone: MobileNetV3 | EMA: {CONFIG['EMA_ENABLED']}")

In [ ]:
# ============================================================================
# CELL 3: METRICS LOGGING (CLEAN & RESUME READY)
# ============================================================================
import csv
from typing import List, Dict, Any
from pathlib import Path
import pandas as pd
import torch
import matplotlib

# Force Matplotlib to non-interactive mode (Agg) to prevent display errors
try:
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

class MetricsLoggerV2:
    def __init__(self, config: Dict, logger):
        self.config = config
        self.logger = logger
        self.output_dir = Path(config['OUTPUT_DIR'])
        self.logs_dir = self.output_dir / "logs"
        self.plots_dir = self.output_dir / "plots"
        self.logs_dir.mkdir(parents=True, exist_ok=True)
        self.plots_dir.mkdir(parents=True, exist_ok=True)
        
        self.epoch_metrics: List[Dict[str, Any]] = []
        self.val_metrics: List[Dict[str, Any]] = []
        
        # 1. Load history (Resume logic)
        self.load_historical_data()
        
        # 2. Sync history to CSV immediately
        self._ensure_csv_files_sync()

    def load_historical_data(self):
        """Auto-detect and load historical logs from Resume dir or Current dir."""
        if pd is None: return

        # Sources
        epoch_source = self.logs_dir / "epoch_metrics.csv"
        val_source = self.logs_dir / "validation_metrics.csv"

        # Check Resume Source if local not found
        if not epoch_source.exists() and self.config.get('RESUME_TRAINING') and self.config.get('RESUME_LOGS_DIR'):
            resume_dir = Path(self.config['RESUME_LOGS_DIR'])
            epoch_source = resume_dir / "epoch_metrics.csv"
            val_source = resume_dir / "validation_metrics.csv"
            self.logger.info(f" Found RESUME log source: {epoch_source}")

        # Load Epoch Data
        try:
            if epoch_source.exists():
                df = pd.read_csv(epoch_source)
                df = df.drop_duplicates(subset=['epoch'], keep='last').sort_values('epoch')
                self.epoch_metrics = df.to_dict('records')
                self.logger.info(f" Loaded {len(self.epoch_metrics)} epoch records.")
        except Exception as e:
            self.logger.warning(f"Failed to load epoch logs: {e}")

        # Load Validation Data
        try:
            if val_source.exists():
                df = pd.read_csv(val_source)
                df = df.drop_duplicates(subset=['epoch'], keep='last').sort_values('epoch')
                self.val_metrics = df.to_dict('records')
                self.logger.info(f" Loaded {len(self.val_metrics)} validation records.")
        except Exception as e:
            self.logger.warning(f"Failed to load validation logs: {e}")

    def _ensure_csv_files_sync(self):
        """Sync loaded history to local CSV files."""
        # Sync Epoch Metrics
        epoch_csv = self.logs_dir / "epoch_metrics.csv"
        fieldnames_epoch = [
            'epoch', 'train_loss', 'learning_rate', 'epoch_time_sec',
            'throughput', 'gpu_memory_mb', 'grad_norm', 'skip_count'
        ]
        self._write_csv(epoch_csv, fieldnames_epoch, self.epoch_metrics)

        # Sync Validation Metrics
        val_csv = self.logs_dir / "validation_metrics.csv"
        fieldnames_val = [
            'epoch', 'mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium',
            'AP_large', 'precision', 'recall', 'f1', 'val_time_sec'
        ]
        self._write_csv(val_csv, fieldnames_val, self.val_metrics)
        
        self.logger.info(" History synchronized to local CSVs.")

    def _write_csv(self, path, fieldnames, data):
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            if data:
                for row in data:
                    filtered_row = {k: row.get(k, 0) for k in fieldnames}
                    writer.writerow(filtered_row)

    def log_epoch(self, epoch: int, metrics: Dict[str, Any]):
        try:
            # Handle GPU memory fallback
            gpu_mem = float(metrics.get('gpu_memory_mb', 0.0))
            if torch.cuda.is_available() and gpu_mem == 0.0:
                gpu_mem = torch.cuda.memory_allocated() / 1024**2
            
            row = {
                'epoch': int(epoch),
                'train_loss': float(metrics.get('train_loss', 0.0)),
                'learning_rate': float(metrics.get('learning_rate', 0.0)),
                'epoch_time_sec': float(metrics.get('epoch_time_sec', 0.0)),
                'throughput': float(metrics.get('throughput', 0.0)),
                'gpu_memory_mb': gpu_mem,
                'grad_norm': float(metrics.get('grad_norm', 0.0)),
                'skip_count': int(metrics.get('skip_count', 0))
            }
            self.epoch_metrics.append(row)
            
            # Append to file
            with open(self.logs_dir / 'epoch_metrics.csv', 'a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=list(row.keys()))
                writer.writerow(row)
            
            print(f" [EPOCH {epoch} SUMMARY] Loss: {row['train_loss']:.4f} | Time: {row['epoch_time_sec']:.1f}s", flush=True)
            
        except Exception as e:
            print(f"Error logging epoch: {e}")

    def log_validation(self, epoch: int, metrics: Dict[str, Any], val_time_sec: float):
        try:
            row = {
                'epoch': int(epoch),
                'mAP': float(metrics.get('mAP', 0.0)),
                'AP50': float(metrics.get('AP50', 0.0)),
                'AP75': float(metrics.get('AP75', 0.0)),
                'AP_small': float(metrics.get('AP_small', 0.0)),
                'AP_medium': float(metrics.get('AP_medium', 0.0)),
                'AP_large': float(metrics.get('AP_large', 0.0)),
                'precision': float(metrics.get('precision', 0.0)),
                'recall': float(metrics.get('recall', 0.0)),
                'f1': float(metrics.get('f1', 0.0)),
                'val_time_sec': float(val_time_sec)
            }
            self.val_metrics.append(row)
            
            # Append to file
            with open(self.logs_dir / 'validation_metrics.csv', 'a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=list(row.keys()))
                writer.writerow(row)
            
            print(f"\n [VAL RESULT E{epoch}] mAP: {row['mAP']:.4f} | AP50: {row['AP50']:.4f} | F1: {row['f1']:.4f}", flush=True)
            print(f"   Details: AP75={row['AP75']:.3f}", flush=True)
            print(f"   Scales : Small={row['AP_small']:.3f} | Medium={row['AP_medium']:.3f} | Large={row['AP_large']:.3f}\n", flush=True)
            
        except Exception as e:
            print(f"Error logging validation: {e}")

# Instantiate
metrics_logger = MetricsLoggerV2(CONFIG, logger)
logger.info("Metrics Logger V2 initialized")

In [ ]:
# ============================================================================
# CELL 4: MEMORY MONITOR (ROBUST)
# ============================================================================
from typing import Optional, Union

class MemoryMonitor:
    """
    Tracks GPU memory usage safely.
    Works on both CPU (no-op) and GPU.
    """
    def __init__(self, device: Union[torch.device, int, None] = 0):
        self.peak_mb = 0.0
        self.current_mb = 0.0
        self.device_idx = None

        if torch.cuda.is_available():
            if isinstance(device, torch.device) and device.type == 'cuda':
                self.device_idx = device.index
            elif isinstance(device, int):
                self.device_idx = device
            else:
                self.device_idx = torch.cuda.current_device()

    def update(self):
        """Update current and peak GPU memory usage."""
        if self.device_idx is None: return

        try:
            # Current reserved memory
            reserved = torch.cuda.memory_reserved(self.device_idx)
            self.current_mb = float(reserved) / (1024 ** 2)
            
            # Peak reserved memory
            if hasattr(torch.cuda, "max_memory_reserved"):
                max_reserved = torch.cuda.max_memory_reserved(self.device_idx)
            else:
                max_reserved = reserved # Fallback for older Torch versions
                
            self.peak_mb = max(self.peak_mb, float(max_reserved) / (1024 ** 2))
        except Exception:
            pass # Silent fail to not interrupt training

    def reset(self):
        """Reset peak stats."""
        self.peak_mb = 0.0
        self.current_mb = 0.0
        if self.device_idx is not None and hasattr(torch.cuda, "reset_peak_memory_stats"):
            try:
                torch.cuda.reset_peak_memory_stats(self.device_idx)
            except Exception:
                pass

    def __repr__(self):
        return f"MemoryMonitor(curr={self.current_mb:.1f}MB, peak={self.peak_mb:.1f}MB)"

# Instantiate
memory_monitor = MemoryMonitor()
logger.info(f"MemoryMonitor initialized: {memory_monitor}")

In [ ]:
# ============================================================================
# CELL 5: SCIENTIFIC VISUALIZATIONS (HARDENED FOR HEADLESS SERVER)
# ============================================================================
import matplotlib
# --- FIX: Force Agg backend to prevent display errors on Kaggle ---
matplotlib.use('Agg') 
from matplotlib import pyplot as plt

# Try to set Times New Roman if available, otherwise leave default
_try_font = "Times New Roman"
try:
    available = matplotlib.font_manager.findfont(_try_font, fallback_to_default=False)
    matplotlib.rcParams['font.family'] = _try_font
except Exception:
    pass

# Global style tuned for publication
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.linewidth'] = 1.5
matplotlib.rcParams['xtick.major.width'] = 1.5
matplotlib.rcParams['ytick.major.width'] = 1.5
matplotlib.rcParams['lines.linewidth'] = 2.0
matplotlib.rcParams['lines.markersize'] = 6

class ScientificVisualizer:
    def __init__(self, output_dir, logger):
        self.output_dir = Path(output_dir)
        self.plots_dir = self.output_dir / 'plots' / 'figures'
        self.plots_dir.mkdir(parents=True, exist_ok=True)
        self.logger = logger
        # color palette (ensure enough colors)
        self.colors = ['#0072B2', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#D55E00', '#2ECC71', '#E74C3C']

    # --- internal helpers -------------------------------------------------
    def _ensure_df_sorted(self, df, epoch_col='epoch'):
        if df is None or df.empty:
            return pd.DataFrame()
        df = df.copy()
        if epoch_col in df.columns:
            df = df.sort_values(epoch_col).reset_index(drop=True)
        return df

    def _safe_series(self, df, col, default=0.0):
        if df is None or df.empty or col not in df.columns:
            return np.array([default] * len(df)) if df is not None else np.array([])
        # convert to numpy and replace NaN with default
        arr = df[col].to_numpy(dtype=float)
        arr = np.nan_to_num(arr, nan=float(default), posinf=float(default), neginf=float(default))
        return arr

    # --- plotting methods -----------------------------------------------
    def plot_loss_convergence(self, epoch_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_loss_convergence")
            return None
        epoch_df = self._ensure_df_sorted(epoch_df)
        if epoch_df.empty:
            self.logger.warning("plot_loss_convergence: empty epoch_df")
            return None
        try:
            epochs = epoch_df['epoch'].to_numpy()
            loss = self._safe_series(epoch_df, 'train_loss', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, loss, color=self.colors[0], linewidth=2.5,
                    label='Training Loss', marker='o', markersize=4,
                    markevery=max(1, len(epochs)//10))
            ax.fill_between(epochs, loss, alpha=0.15, color=self.colors[0])
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
            ax.set_title('Training Loss Convergence', fontsize=13, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='upper right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_01_loss_convergence.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_loss_convergence failed: {e}")
            return None

    def plot_learning_rate_schedule(self, epoch_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_learning_rate_schedule")
            return None
        epoch_df = self._ensure_df_sorted(epoch_df)
        if epoch_df.empty:
            self.logger.warning("plot_learning_rate_schedule: empty epoch_df")
            return None
        try:
            epochs = epoch_df['epoch'].to_numpy()
            lr = self._safe_series(epoch_df, 'learning_rate', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.semilogy(epochs, lr, color=self.colors[1], linewidth=2.5,
                        marker='s', markersize=4, markevery=max(1, len(epochs)//10), label='Learning Rate')
            ax.fill_between(epochs, lr, alpha=0.12, color=self.colors[1])
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
            ax.set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, which='both')
            ax.legend(fontsize=11, loc='best', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_02_learning_rate.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_learning_rate_schedule failed: {e}")
            return None

    def plot_detection_metrics(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_detection_metrics")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_detection_metrics: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            mAP = self._safe_series(val_df, 'mAP', default=0.0)
            AP50 = self._safe_series(val_df, 'AP50', default=0.0)
            AP75 = self._safe_series(val_df, 'AP75', default=0.0)
            fig, ax = plt.subplots(figsize=(10, 7))
            ax.plot(epochs, mAP, color=self.colors[0], linewidth=2.5, marker='o', markersize=5, label='mAP@IoU=0.50:0.95')
            ax.plot(epochs, AP50, color=self.colors[1], linewidth=2.0, marker='s', markersize=5, label='AP@IoU=0.50')
            ax.plot(epochs, AP75, color=self.colors[2], linewidth=2.0, marker='^', markersize=5, label='AP@IoU=0.75')
            ax.fill_between(epochs, mAP, alpha=0.1, color=self.colors[0])
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Average Precision', fontsize=12, fontweight='bold')
            ax.set_title('Detection Performance at Different IoU Thresholds', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True, ncol=1)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_03_detection_metrics.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_detection_metrics failed: {e}")
            return None

    def plot_scale_analysis(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_scale_analysis")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_scale_analysis: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            AP_small = self._safe_series(val_df, 'AP_small', default=0.0)
            AP_medium = self._safe_series(val_df, 'AP_medium', default=0.0)
            AP_large = self._safe_series(val_df, 'AP_large', default=0.0)
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot(epochs, AP_small, color=self.colors[3], linewidth=2.5, marker='o', markersize=5, label='Small Objects')
            ax.plot(epochs, AP_medium, color=self.colors[4], linewidth=2.5, marker='s', markersize=5, label='Medium Objects')
            ax.plot(epochs, AP_large, color=self.colors[5], linewidth=2.5, marker='^', markersize=5, label='Large Objects')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Average Precision', fontsize=12, fontweight='bold')
            ax.set_title('Scale-Dependent Performance', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_04_scale_analysis.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_scale_analysis failed: {e}")
            return None

    def plot_precision_curve(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_precision_curve")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_precision_curve: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            precision = self._safe_series(val_df, 'precision', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, precision, color=self.colors[6], linewidth=2.5, marker='o', markersize=5, label='Precision')
            ax.fill_between(epochs, precision, alpha=0.15, color=self.colors[6])
            if precision.size > 0:
                best_prec_idx = int(np.nanargmax(precision))
                ax.scatter(epochs[best_prec_idx], precision[best_prec_idx], color='red', s=120, marker='*', zorder=5,
                           label=f'Best: {precision[best_prec_idx]:.4f}')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
            ax.set_title('Precision Evolution During Training', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_05_precision.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_precision_curve failed: {e}")
            return None

    def plot_recall_curve(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_recall_curve")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_recall_curve: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            recall = self._safe_series(val_df, 'recall', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, recall, color=self.colors[7], linewidth=2.5, marker='s', markersize=5, label='Recall')
            ax.fill_between(epochs, recall, alpha=0.15, color=self.colors[7])
            if recall.size > 0:
                best_recall_idx = int(np.nanargmax(recall))
                ax.scatter(epochs[best_recall_idx], recall[best_recall_idx], color='red', s=120, marker='*', zorder=5,
                           label=f'Best: {recall[best_recall_idx]:.4f}')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('Recall', fontsize=12, fontweight='bold')
            ax.set_title('Recall Evolution During Training', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_06_recall.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_recall_curve failed: {e}")
            return None

    def plot_f1_score(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_f1_score")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_f1_score: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            f1 = self._safe_series(val_df, 'f1', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, f1, color=self.colors[2], linewidth=2.5, marker='D', markersize=5, label='F1 Score')
            ax.fill_between(epochs, f1, alpha=0.15, color=self.colors[2])
            if f1.size > 0:
                best_f1_idx = int(np.nanargmax(f1))
                ax.scatter(epochs[best_f1_idx], f1[best_f1_idx], color='red', s=120, marker='*', zorder=5,
                           label=f'Best: {f1[best_f1_idx]:.4f}')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
            ax.set_title('F1 Score Evolution During Training', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_07_f1_score.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_f1_score failed: {e}")
            return None

    def plot_map_per_epoch(self, val_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping plot_map_per_epoch")
            return None
        val_df = self._ensure_df_sorted(val_df)
        if val_df.empty:
            self.logger.warning("plot_map_per_epoch: empty val_df")
            return None
        try:
            epochs = val_df['epoch'].to_numpy()
            maps = self._safe_series(val_df, 'mAP', default=0.0)
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.plot(epochs, maps, color=self.colors[5], linewidth=2.5, marker='o', markersize=5, label='mAP per epoch')
            ax.fill_between(epochs, maps, alpha=0.15, color=self.colors[5])
            if maps.size > 0:
                best_map_idx = int(np.nanargmax(maps))
                ax.scatter(epochs[best_map_idx], maps[best_map_idx], color='red', s=120, marker='*', zorder=5,
                           label=f'Best: {maps[best_map_idx]:.4f}')
            ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            ax.set_ylabel('mAP', fontsize=12, fontweight='bold')
            ax.set_title('Validation mAP per Epoch', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
            ax.legend(fontsize=11, loc='lower right', frameon=True)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            savepath = self.plots_dir / 'fig_08_map_per_epoch.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Figure saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"plot_map_per_epoch failed: {e}")
            return None

    def create_results_table(self, val_df, epoch_df):
        if plt is None:
            self.logger.warning("Matplotlib not available; skipping create_results_table")
            return None
        val_df = self._ensure_df_sorted(val_df)
        epoch_df = self._ensure_df_sorted(epoch_df)
        if val_df.empty:
            self.logger.warning("create_results_table: empty val_df")
            return None
        try:
            fig, ax = plt.subplots(figsize=(12, 4))
            ax.axis('tight')
            ax.axis('off')
            def get_best_info(df, col):
                if col not in df.columns or df.empty:
                    return ("N/A", "N/A", "N/A")
                best_idx = int(np.nanargmax(df[col].to_numpy(dtype=float)))
                best_val = float(df.iloc[best_idx][col])
                best_epoch = int(df.iloc[best_idx]['epoch']) if 'epoch' in df.columns else "N/A"
                final_val = float(df[col].iloc[-1]) if len(df) > 0 else "N/A"
                return (f"{best_val:.4f}", f"{best_epoch}", f"{final_val:.4f}")
            results_data = [
                ['Metric', 'Best Value', 'Epoch', 'Final Value'],
                ['mAP@50:95', *get_best_info(val_df, 'mAP')],
                ['AP@50', *get_best_info(val_df, 'AP50')],
                ['AP@75', *get_best_info(val_df, 'AP75')],
                ['F1 Score', *get_best_info(val_df, 'f1')],
            ]
            table = ax.table(cellText=results_data, cellLoc='center', loc='center', colWidths=[0.25]*4)
            table.auto_set_font_size(False)
            table.set_fontsize(11)
            table.scale(1, 2.0)
            # header styling
            for i in range(len(results_data[0])):
                cell = table[(0, i)]
                cell.set_facecolor('#0072B2')
                cell.get_text().set_color('white')
                cell.get_text().set_weight('bold')
            # zebra rows
            for i in range(1, len(results_data)):
                for j in range(len(results_data[0])):
                    if i % 2 == 0:
                        table[(i, j)].set_facecolor('#F0F0F0')
                    else:
                        table[(i, j)].set_facecolor('white')
            plt.title('Detection Results Summary', fontsize=13, fontweight='bold', pad=12)
            plt.tight_layout()
            savepath = self.plots_dir / 'table_01_results_summary.png'
            fig.savefig(savepath, dpi=300, bbox_inches='tight')  # Increased DPI for publication
            plt.close(fig)
            self.logger.info(f"Table saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"create_results_table failed: {e}")
            return None

    def generate_all_figures(self, epoch_df, val_df):
        self.logger.info("Generating publication-quality figures...")
        figures = []
        epoch_df = self._ensure_df_sorted(epoch_df)
        val_df = self._ensure_df_sorted(val_df)
        if not epoch_df.empty:
            figures.append(self.plot_loss_convergence(epoch_df))
            figures.append(self.plot_learning_rate_schedule(epoch_df))
        if not val_df.empty:
            figures.append(self.plot_detection_metrics(val_df))
            figures.append(self.plot_scale_analysis(val_df))
            figures.append(self.plot_precision_curve(val_df))
            figures.append(self.plot_recall_curve(val_df))
            figures.append(self.plot_f1_score(val_df))
            figures.append(self.plot_map_per_epoch(val_df))
        if not epoch_df.empty and not val_df.empty:
            figures.append(self.create_results_table(val_df, epoch_df))
        self.logger.info("All figures generated")
        self.logger.info(f"Output directory: {self.plots_dir}")
        return [f for f in figures if f is not None]

# convenience function
def generate_publication_figures(config, logger):
    visualizer = ScientificVisualizer(config['OUTPUT_DIR'], logger)
    epoch_csv = Path(config['OUTPUT_DIR']) / 'logs' / 'epoch_metrics.csv'
    val_csv = Path(config['OUTPUT_DIR']) / 'logs' / 'validation_metrics.csv'
    epoch_df = pd.read_csv(epoch_csv) if epoch_csv.exists() else pd.DataFrame()
    val_df = pd.read_csv(val_csv) if val_csv.exists() else pd.DataFrame()
    return visualizer.generate_all_figures(epoch_df, val_df)

logger.info("Scientific Visualizer loaded")

In [ ]:
# ============================================================================
# CELL 6: DATASET - FINAL (ROBUST & SANITIZED)
# ============================================================================
from typing import Optional, List, Dict, Tuple
from pathlib import Path
import numpy as np
import torch
import cv2
import logging

logger = logging.getLogger(__name__)

# --- Safe Imports ---
try:
    cv2.setNumThreads(0)
except Exception:
    pass

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    from pycocotools.coco import COCO
except ImportError:
    print("[WARN] Libraries missing in Cell 6. Dataset might fail.")

def get_transforms(train: bool = True, input_size: int = 448, include_bbox: bool = True):
    """
    Albumentations pipeline:
    - LongestMaxSize: Resize cạnh lớn nhất về input_size (giữ aspect ratio).
    - PadIfNeeded: Pad cạnh còn lại để được ảnh vuông (input_size x input_size).
    - Normalize: Chuẩn hóa theo ImageNet mean/std.
    """
    common_transforms = [
        A.LongestMaxSize(max_size=input_size),
        A.PadIfNeeded(min_height=input_size, min_width=input_size, 
                      border_mode=cv2.BORDER_CONSTANT, value=0), 
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]
    
    if include_bbox:
        bbox_params = A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.1)
        if train:
            aug = [
                A.HorizontalFlip(p=0.5),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
                A.Blur(blur_limit=3, p=0.1),
                A.RandomBrightnessContrast(p=0.2),
            ]
            return A.Compose(aug + common_transforms, bbox_params=bbox_params)
        return A.Compose(common_transforms, bbox_params=bbox_params)
    else:
        return A.Compose(common_transforms)

class MultiScaleTrafficDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir: str, ann_file: str, config: Dict, train: bool = True):
        super().__init__()
        self.img_dir = Path(img_dir)
        self.coco = COCO(str(ann_file))
        self.config = config
        self.train = train
        self.current_size = config.get('BASE_INPUT_SIZE', 448)
        self.min_box_size = config.get('MIN_BOX_SIZE', 2.0) 
        
        self.cat_ids = sorted(self.coco.getCatIds())
        self.catid2label = {cid: i+1 for i, cid in enumerate(self.cat_ids)}
        
        # Filter valid images (must have annotations)
        self.valid_img_ids = []
        all_ids = self.coco.getImgIds()
        for img_id in all_ids:
            anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
            if len(anns) > 0:
                self.valid_img_ids.append(img_id)
        
        self.error_count = 0 
        logger.info(f"Dataset loaded: {len(self.valid_img_ids)} valid images.")

    def __len__(self):
        return len(self.valid_img_ids)

    def _load_safe_image(self, img_path):
        try:
            image = cv2.imread(img_path)
            if image is None: raise ValueError("Image None")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            return image
        except Exception:
            # Return black image if load fails to prevent crash
            return np.zeros((self.current_size, self.current_size, 3), dtype=np.uint8)

    def __getitem__(self, idx):
        cv2.setNumThreads(0)
        
        img_id = self.valid_img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = str(self.img_dir / img_info['file_name'])
        
        image = self._load_safe_image(img_path)
        orig_h, orig_w = image.shape[:2]

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        
        raw_bboxes, raw_labels = [], []
        
        # --- 1. Load & Sanitize Annotations (Clamp to image boundaries) ---
        for ann in anns:
            if ann['category_id'] not in self.catid2label: continue
            bbox = ann.get('bbox')
            if not bbox or len(bbox) < 4: continue
            
            x, y, w, h = float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])
            
            # Clamp box to image dimensions
            x1 = max(0, min(orig_w - 1, x))
            y1 = max(0, min(orig_h - 1, y))
            x2 = max(x1 + 1, min(orig_w, x + w))
            y2 = max(y1 + 1, min(orig_h, y + h))
            
            new_w, new_h = x2 - x1, y2 - y1
            
            # Filter tiny boxes
            if new_w < self.min_box_size or new_h < self.min_box_size: continue
            
            # Albumentations expects [x, y, w, h]
            raw_bboxes.append([x1, y1, new_w, new_h])
            raw_labels.append(self.catid2label[ann['category_id']])

        # --- 2. Apply Augmentations ---
        try:
            transform = get_transforms(self.train, self.current_size, include_bbox=True)
            transformed = transform(image=image, bboxes=raw_bboxes, labels=raw_labels)
            image_t = transformed['image']
            bboxes_t = transformed['bboxes']
            labels_t = transformed['labels']

        except Exception as e:
            if self.error_count < 5: 
                logger.warning(f"Transform fail ID {img_id}: {e}. Fallback to resize only.")
            self.error_count += 1
            
            # Fallback: Resize image directly, drop all boxes (Negative sample)
            image_resized = cv2.resize(image, (self.current_size, self.current_size))
            img_float = image_resized.astype(np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img_norm = (img_float - mean) / std
            image_t = torch.from_numpy(img_norm.transpose(2, 0, 1)).float()
            bboxes_t, labels_t = [], []

        # --- 3. Final Format (XYXY) for Model ---
        valid_boxes, valid_labels = [], []
        for i, box in enumerate(bboxes_t):
            x, y, w, h = box
            # Clamp to current_size (transformed size)
            x1 = max(0, min(self.current_size, x))
            y1 = max(0, min(self.current_size, y))
            x2 = max(x1, min(self.current_size, x + w))
            y2 = max(y1, min(self.current_size, y + h))
            
            if (x2 - x1) > 1.0 and (y2 - y1) > 1.0: 
                 valid_boxes.append([x1, y1, x2, y2]) 
                 valid_labels.append(labels_t[i])

        if len(valid_boxes) > 0:
            boxes_final = torch.tensor(valid_boxes, dtype=torch.float32)
            labels_final = torch.tensor(valid_labels, dtype=torch.int64)
        else:
            boxes_final = torch.zeros((0, 4), dtype=torch.float32)
            labels_final = torch.zeros((0,), dtype=torch.int64)

        target = {
            'boxes': boxes_final,
            'labels': labels_final,
            'image_id': torch.tensor([img_id]),
            'orig_size': torch.tensor([orig_h, orig_w]), # Critical for rescaling
            'img_size': torch.tensor([self.current_size, self.current_size])
        }
        
        return image_t, target
        
    def set_input_size(self, size):
        self.current_size = size

logger.info("MultiScaleTrafficDataset loaded (Sanitized & Optimized)")

In [ ]:
# ============================================================================
# CELL 7: MODEL BUILDING (MobileNetV3 + CIoU Loss + EMA) - FINAL FIX
# ============================================================================
import math
import torch
import torch.nn as nn
from copy import deepcopy
import logging
from typing import List, Dict, Tuple

# Torchvision imports
try:
    from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
    from torchvision.models.detection import RetinaNet
    from torchvision.models.detection.anchor_utils import AnchorGenerator
    from torchvision.models.detection.backbone_utils import BackboneWithFPN
    from torchvision.models.detection.retinanet import RetinaNetHead
    from torchvision.ops.feature_pyramid_network import LastLevelP6P7
    from torchvision.models._utils import IntermediateLayerGetter
    from torchvision.ops import sigmoid_focal_loss, complete_box_iou_loss
    from torchvision.ops import boxes as box_ops
except ImportError:
    print("[ERROR] torchvision not found or version too old.")

logger = logging.getLogger(__name__)

# ----------------------------------------------------------------------------
# 1. CUSTOM RETINANET WITH CIoU LOSS
# ----------------------------------------------------------------------------
class CIoURetinaNet(RetinaNet):
    """
    Custom RetinaNet that uses CIoU Loss for regression.
    Includes manual box decoding to avoid shape mismatch issues.
    """
    def compute_loss(self, targets, head_outputs, anchors):
        matched_idxs = []
        for anchors_per_image, targets_per_image in zip(anchors, targets):
            if targets_per_image["boxes"].numel() == 0:
                matched_idxs.append(
                    torch.full((anchors_per_image.size(0),), -1, dtype=torch.int64, device=anchors_per_image.device)
                )
                continue

            match_quality_matrix = box_ops.box_iou(targets_per_image["boxes"], anchors_per_image)
            matched_idxs.append(self.proposal_matcher(match_quality_matrix))

        return self.compute_loss_ciou(targets, head_outputs, anchors, matched_idxs)

    def decode_boxes(self, rel_codes, boxes):
        """
        Manual implementation of box decoding (delta -> xyxy).
        rel_codes: [N, 4] (dx, dy, dw, dh)
        boxes: [N, 4] (x1, y1, x2, y2) -> Anchors
        """
        # Unpack anchors
        wx, wy, ww, wh = (1.0, 1.0, 1.0, 1.0) # Weights (default torchvision)
        
        dx = rel_codes[:, 0] / wx
        dy = rel_codes[:, 1] / wy
        dw = rel_codes[:, 2] / ww
        dh = rel_codes[:, 3] / wh

        # Anchor width/height/center
        ex_widths = boxes[:, 2] - boxes[:, 0]
        ex_heights = boxes[:, 3] - boxes[:, 1]
        ex_ctr_x = boxes[:, 0] + 0.5 * ex_widths
        ex_ctr_y = boxes[:, 1] + 0.5 * ex_heights

        # Apply deltas
        pred_ctr_x = dx * ex_widths + ex_ctr_x
        pred_ctr_y = dy * ex_heights + ex_ctr_y
        pred_w = torch.exp(dw) * ex_widths
        pred_h = torch.exp(dh) * ex_heights

        # Convert back to x1, y1, x2, y2
        pred_boxes1 = pred_ctr_x - 0.5 * pred_w
        pred_boxes2 = pred_ctr_y - 0.5 * pred_h
        pred_boxes3 = pred_ctr_x + 0.5 * pred_w
        pred_boxes4 = pred_ctr_y + 0.5 * pred_h
        
        return torch.stack((pred_boxes1, pred_boxes2, pred_boxes3, pred_boxes4), dim=1)

    def compute_loss_ciou(self, targets, head_outputs, anchors, matched_idxs):
        # 1. Flatten Head Outputs
        if isinstance(head_outputs["cls_logits"], list):
            cls_logits = torch.cat(head_outputs["cls_logits"], dim=1)
            bbox_regression = torch.cat(head_outputs["bbox_regression"], dim=1)
        else:
            cls_logits = head_outputs["cls_logits"]
            bbox_regression = head_outputs["bbox_regression"]

        # Flatten (N * A, C) and (N * A, 4)
        cls_logits = cls_logits.view(-1, cls_logits.shape[-1])
        bbox_regression = bbox_regression.view(-1, 4)

        # 2. Flatten Anchors & Matches
        matched_idxs = torch.cat(matched_idxs, dim=0)
        flatten_anchors = torch.cat(anchors, dim=0)

        # 3. Prepare Classification Targets
        gt_classes_target = []
        for targets_per_image, matched_idxs_per_image in zip(targets, matched_idxs.split([a.size(0) for a in anchors])):
            if targets_per_image["boxes"].numel() == 0:
                device = targets_per_image["boxes"].device
                labeled_matched_idxs = torch.full((matched_idxs_per_image.shape[0],), -1, dtype=torch.int64, device=device)
            else:
                labeled_matched_idxs = matched_idxs_per_image.clamp(min=0)
                labels = targets_per_image["labels"][labeled_matched_idxs]
                labels = labels.to(dtype=torch.int64)
                bg_indices = matched_idxs_per_image < 0
                labels[bg_indices] = 0 
                labeled_matched_idxs = labels
            gt_classes_target.append(labeled_matched_idxs)

        gt_classes_target = torch.cat(gt_classes_target, dim=0)
        valid_idxs = gt_classes_target >= 0
        foreground_idxs = gt_classes_target > 0
        
        num_foreground = foreground_idxs.sum()
        num_foreground = max(1, num_foreground)

        # Focal Loss (Classification)
        num_classes = cls_logits.shape[-1]
        gt_classes_target_one_hot = torch.nn.functional.one_hot(gt_classes_target[valid_idxs], num_classes=num_classes)
        target_one_hot_foreground = gt_classes_target_one_hot[:, 1:].float()
        logits_foreground = cls_logits[valid_idxs][:, 1:] 

        loss_cls = sigmoid_focal_loss(
            logits_foreground,
            target_one_hot_foreground,
            alpha=0.25,
            gamma=2.0,
            reduction="sum",
        ) / num_foreground

        # 4. Regression Loss (CIoU Loss)
        if foreground_idxs.sum() > 0:
            # Lấy anchor và deltas dự đoán cho foreground
            pos_anchors = flatten_anchors[foreground_idxs]
            pred_deltas = bbox_regression[foreground_idxs]
            
            # --- FIX: Decode thủ công (Tránh lỗi shape của torchvision) ---
            pred_boxes = self.decode_boxes(pred_deltas, pos_anchors)

            # Lấy GT Boxes tương ứng
            matched_split = matched_idxs.split([a.size(0) for a in anchors])
            foreground_split = foreground_idxs.split([a.size(0) for a in anchors])
            
            target_boxes_list = []
            for targets_per_image, matched_idxs_per_image, fg_idxs_per_image in zip(targets, matched_split, foreground_split):
                if targets_per_image["boxes"].numel() == 0: continue
                # Lấy index GT của các anchor dương tính
                pos_gt_idxs = matched_idxs_per_image[fg_idxs_per_image]
                target_boxes_list.append(targets_per_image["boxes"][pos_gt_idxs])
            
            if len(target_boxes_list) > 0:
                target_boxes = torch.cat(target_boxes_list, dim=0)
                # Tính CIoU Loss
                loss_bbox = complete_box_iou_loss(pred_boxes, target_boxes, reduction="sum") / num_foreground
            else:
                loss_bbox = torch.tensor(0.0, device=bbox_regression.device)
        else:
            loss_bbox = torch.tensor(0.0, device=bbox_regression.device)

        return {"classification": loss_cls, "bbox_regression": loss_bbox}

# ----------------------------------------------------------------------------
# 2. MODEL UTILITIES
# ----------------------------------------------------------------------------
def freeze_batchnorm_layers(model):
    frozen_count = 0
    for module in model.modules():
        if isinstance(module, torch.nn.BatchNorm2d):
            module.eval() 
            for param in module.parameters():
                param.requires_grad = False
            frozen_count += 1
    logger.info(f"MODEL: Froze {frozen_count} BatchNorm layers.")

def get_mobilenet_backbone():
    weights = MobileNet_V3_Large_Weights.DEFAULT
    backbone = mobilenet_v3_large(weights=weights)
    return_layers = {'6': '0', '12': '1', '16': '2'}
    in_channels_list = [40, 112, 960]
    out_channels = 256
    body = IntermediateLayerGetter(backbone.features, return_layers=return_layers)
    backbone_with_fpn = BackboneWithFPN(
        body, return_layers=return_layers, in_channels_list=in_channels_list,
        out_channels=out_channels, extra_blocks=LastLevelP6P7(out_channels, out_channels) 
    )
    return backbone_with_fpn

# ----------------------------------------------------------------------------
# 3. BUILDER FUNCTION (CIoURetinaNet)
# ----------------------------------------------------------------------------
def build_model(num_classes: int, device: torch.device, prior_prob: float = 0.01):
    logger.info("MODEL: Building Custom RetinaNet (MobileNetV3 + CIoU)...")
    
    backbone = get_mobilenet_backbone()
    anchor_sizes = ((16, 24, 32), (32, 48, 64), (64, 96, 128), (128, 192, 256), (256, 320, 400))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)
    
    head = RetinaNetHead(
        in_channels=backbone.out_channels,
        num_anchors=anchor_generator.num_anchors_per_location()[0],
        num_classes=num_classes
    )
    
    # Use custom CIoURetinaNet
    model = CIoURetinaNet(
        backbone=backbone,
        num_classes=num_classes,
        anchor_generator=anchor_generator,
        head=head,
        detections_per_img=100,
        score_thresh=0.05
    )

    try:
        prior_bias = -math.log((1.0 - prior_prob) / prior_prob)
        for name, param in model.head.classification_head.named_parameters():
            if "cls_logits" in name and "bias" in name:
                torch.nn.init.constant_(param, prior_bias)
    except Exception as e:
        logger.warning(f"Init bias warning: {e}")

    freeze_batchnorm_layers(model)
    model = model.to(device)
    
    return model

# ----------------------------------------------------------------------------
# 4. EMA UTILITY
# ----------------------------------------------------------------------------
class ModelEMA:
    def __init__(self, model, decay=0.9999, updates=0):
        self.ema = deepcopy(model).eval() 
        self.updates = updates
        self.decay = lambda x: decay * (1 - math.exp(-x / 2000))
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            self.updates += 1
            d = self.decay(self.updates)
            msd = model.state_dict()
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v *= d
                    v += (1.0 - d) * msd[k].detach()

logger.info("Custom CIoURetinaNet Builder Loaded (FINAL FIX)")

In [ ]:
# ============================================================================
# CELL 8: VALIDATION FUNCTION (FIXED PRECISION & RECALL LOGGING)
# ============================================================================

import time
import json
import numpy as np
import torch
from tqdm.notebook import tqdm
import io
from contextlib import redirect_stdout

def validate_model(
    model, val_loader, device, config: Dict, logger,
    category_id_map: Optional[Dict[int, int]] = None,
    save_predictions_path: Optional[str] = None
):
    model.eval()
    print("\n--- Starting Validation (Rescale Mode) ---", flush=True)
    start_time = time.time()

    try:
        coco_gt = COCO(str(config['VAL_ANN']))
    except Exception as e:
        print(f"Error loading COCO: {e}", flush=True)
        return {'mAP': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'inference_time': 0.0}

    all_predictions = []
    score_thresh = 0.01

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            images, targets = batch
            if images is None or targets is None: continue
            
            if isinstance(images, (list, tuple)): images = [img.to(device) for img in images]
            else: images = images.to(device)

            try:
                outputs = model(images)
            except: continue

            for i, out in enumerate(outputs):
                tgt = targets[i]
                img_id = int(tgt.get('image_id').item()) if isinstance(tgt, dict) and 'image_id' in tgt else 0
                
                # --- LOGIC RESCALE MỚI ---
                if 'orig_size' in tgt and 'img_size' in tgt:
                    orig_h, orig_w = tgt['orig_size'].tolist()
                    curr_h, curr_w = tgt['img_size'].tolist()
                    scale_x = orig_w / curr_w
                    scale_y = orig_h / curr_h
                else:
                    scale_x, scale_y = 1.0, 1.0

                boxes = out['boxes'].cpu().numpy()
                scores = out['scores'].cpu().numpy()
                labels = out['labels'].cpu().numpy()

                keep = scores > score_thresh
                boxes = boxes[keep]
                scores = scores[keep]
                labels = labels[keep]

                if len(boxes) == 0: continue

                for box, score, label in zip(boxes, scores, labels):
                    x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
                    
                    # Rescale
                    x1 *= scale_x
                    y1 *= scale_y
                    x2 *= scale_x
                    y2 *= scale_y
                    
                    w, h = x2 - x1, y2 - y1
                    if w <= 0 or h <= 0: continue
                    
                    cat_id = int(label)
                    if category_id_map: cat_id = category_id_map.get(cat_id, cat_id)

                    all_predictions.append({
                        'image_id': img_id,
                        'category_id': cat_id,
                        'bbox': [x1, y1, w, h],
                        'score': float(score)
                    })

    inference_time = time.time() - start_time
    
    # --- KHỞI TẠO ĐẦY ĐỦ KEYS ---
    metrics = {
        'mAP': 0.0, 'AP50': 0.0, 'AP75': 0.0, 
        'AP_small': 0.0, 'AP_medium': 0.0, 'AP_large': 0.0,
        'f1': 0.0, 'precision': 0.0, 'recall': 0.0, # <--- Đã thêm key này
        'inference_time': inference_time
    }

    if not all_predictions:
        print("⚠️ WARNING: No predictions above threshold.", flush=True)
        return metrics

    try:
        with redirect_stdout(io.StringIO()):
            coco_dt = coco_gt.loadRes(all_predictions)
            coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
            coco_eval.evaluate()
            coco_eval.accumulate()
            coco_eval.summarize()

        stats = coco_eval.stats
        # Map các chỉ số COCO sang metrics
        metrics['mAP'] = float(stats[0])       # AP @ IoU=0.50:0.95
        metrics['AP50'] = float(stats[1])      # AP @ IoU=0.50 (Dùng làm Precision)
        metrics['AP75'] = float(stats[2])
        metrics['AP_small'] = float(stats[3])
        metrics['AP_medium'] = float(stats[4])
        metrics['AP_large'] = float(stats[5])
        
        # --- FIX: Gán giá trị Precision & Recall ---
        # Trong object detection, AP50 thường được coi là Precision tổng quát
        metrics['precision'] = float(stats[1]) 
        # stats[8] là Average Recall @ IoU=0.50:0.95 | maxDets=100
        metrics['recall'] = float(stats[8])    

        if metrics['mAP'] > 0:
             metrics['f1'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'] + 1e-6)
        
        print(f" Evaluated {len(all_predictions)} preds. mAP={metrics['mAP']:.4f} | Prec={metrics['precision']:.4f} | Rec={metrics['recall']:.4f}", flush=True)

    except Exception as e:
        print(f"COCO Eval Error: {e}", flush=True)

    return metrics

In [ ]:
# ============================================================================
# CELL 9: TRAINING LOOP (WITH EMA & MULTI-SCALE)
# ============================================================================
import gc
import time
import random
import numpy as np
from contextlib import nullcontext
from typing import Optional
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from tqdm.notebook import tqdm

def train_one_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
    epoch: int,
    scaler: Optional[GradScaler],
    scheduler=None,
    config: dict = {},
    metrics_logger=None,
    ema=None
):
    logger.info(f"[EPOCH {epoch}] Starting training... (EMA: {ema is not None})")
    
    model.train()
    mem_monitor = MemoryMonitor()
    mem_monitor.reset()

    losses = []
    skip_count = 0
    start_time = time.time()
    
    # --- Multi-Scale Config ---
    multi_scale = config.get('MULTI_SCALE', False)
    input_sizes = config.get('INPUT_SIZES', [448])
    scale_freq = config.get('SCALE_CHANGE_FREQUENCY', 200)
    current_size = config.get('BASE_INPUT_SIZE', 448)

    # Set initial size
    if multi_scale and hasattr(dataloader.dataset, 'set_input_size'):
        dataloader.dataset.set_input_size(current_size)

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}", leave=True)
    accumulation_steps = int(config.get('ACCUMULATION_STEPS', 1))
    grad_clip_value = float(config.get('GRAD_CLIP', 1.0))
    use_amp = config.get('USE_AMP', False)
    amp_ctx = autocast(enabled=use_amp) if use_amp else nullcontext()

    for step, batch in enumerate(pbar, 1):
        # --- Multi-Scale Adjustment ---
        if multi_scale and step % scale_freq == 0:
            current_size = random.choice(input_sizes)
            if hasattr(dataloader.dataset, 'set_input_size'):
                dataloader.dataset.set_input_size(current_size)

        if not batch: continue
        
        try:
            images, targets = batch
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        except Exception:
            continue

        # --- FORWARD PASS ---
        with amp_ctx:
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            
            # Loss Guard (NaN/Inf check)
            if not torch.isfinite(loss):
                optimizer.zero_grad()
                skip_count += 1
                continue
            
            loss = loss / accumulation_steps

        # --- BACKWARD PASS ---
        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # --- OPTIMIZER STEP ---
        if step % accumulation_steps == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_value)

            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
                
            # --- EMA UPDATE ---
            if ema:
                ema.update(model)

            optimizer.zero_grad()
            if scheduler: scheduler.step()

        # Logging
        current_loss = loss.item() * accumulation_steps
        losses.append(current_loss)
        mem_monitor.update()
        
        if step % config.get('LOG_BATCH_INTERVAL', 50) == 0:
            avg_loss = np.mean(losses[-50:])
            lr = optimizer.param_groups[0]['lr']
            pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'sz': f'{current_size}', 'mem': f'{mem_monitor.current_mb:.0f}M'})
            print(f"[E{epoch}|{step}] Loss: {avg_loss:.4f} | LR: {lr:.2e} | Size: {current_size}", flush=True)

    # Epoch Summary
    epoch_time = time.time() - start_time
    avg_loss = float(np.mean(losses)) if losses else 0.0
    throughput = (len(dataloader.dataset) / epoch_time) if epoch_time > 0 else 0.0
    
    metrics = {
        'train_loss': avg_loss,
        'learning_rate': optimizer.param_groups[0]['lr'],
        'epoch_time_sec': epoch_time,
        'throughput': throughput,
        'gpu_memory_mb': mem_monitor.peak_mb,
        'skip_count': skip_count
    }

    if metrics_logger: metrics_logger.log_epoch(epoch, metrics)
    
    # Cleanup
    gc.collect()
    torch.cuda.empty_cache()
    
    return metrics

In [ ]:
# ============================================================================
# CELL 10: TEST EVALUATOR (FIXED VARIABLE NAME ERROR)
# ============================================================================
import os
import time
import json
import logging
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional
import numpy as np
import torch
import cv2
from PIL import Image
from tqdm.notebook import tqdm
import io
from contextlib import redirect_stdout

# Conditional Import for COCO tools
try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except ImportError:
    COCO = None
    COCOeval = None

class ComprehensiveEvaluator:
    """
    Evaluates model on the Test set.
    Uses 'Distorted Resize' strategy to match Validation logic in Cell 8.
    """
    
    def __init__(self, test_img_dir, test_ann_file, logger, config):
        self.test_img_dir = test_img_dir
        self.test_ann_file = test_ann_file
        self.logger = logger
        self.config = config
        self.output_dir = Path(config['OUTPUT_DIR'])
        
        try:
            self.coco_test = COCO(str(test_ann_file))
            self.test_ids = self.coco_test.getImgIds()
            self.logger.info(f"TEST: Loaded {len(self.test_ids)} test images")
        except Exception as e:
            self.logger.error(f"TEST: Failed to load COCO: {e}")
            self.coco_test = None
            self.test_ids = []
    
    def evaluate_on_test_set(self, model, device, classnames, iou_threshold=0.5, score_threshold=0.01, category_mapping=None):
        if not self.coco_test:
            self.logger.warning("TEST: COCO object not initialized.")
            return None
        
        model.eval()
        model = model.to(device)
        
        all_predictions = []
        all_ground_truth = []
        inference_times = []
        
        self.logger.info("=" * 80)
        self.logger.info("TEST: Starting evaluation...")
        
        target_size = self.config.get('BASE_INPUT_SIZE', 448)
        
        with torch.no_grad():
            # Use tqdm for progress tracking
            pbar = tqdm(self.test_ids, desc="Test Eval", leave=False)
            
            for img_id in pbar:
                img_info = self.coco_test.loadImgs(img_id)[0]
                img_path = os.path.join(self.test_img_dir, img_info['file_name'])
                
                if not os.path.exists(img_path): 
                    continue

                # --- 1. Load Image ---
                try:
                    # Use PIL to safely get original dimensions
                    image_pil = Image.open(img_path).convert('RGB')
                    orig_w, orig_h = image_pil.size
                    image_np = np.array(image_pil)
                except Exception:
                    continue
                
                # --- 2. Preprocess (Distorted Resize to match Val Logic) ---
                image_tensor = self._prepare_input(image_np, device, target_size)
                
                # --- 3. Inference ---
                t0 = time.time()
                try:
                    outputs = model([image_tensor])
                except Exception:
                    continue
                inference_time = (time.time() - t0) * 1000 # ms
                inference_times.append(inference_time)
                
                output = outputs[0]
                boxes = output['boxes'].cpu().numpy()
                scores = output['scores'].cpu().numpy()
                labels = output['labels'].cpu().numpy()
                
                # Calculate Scale Factors (Distorted)
                scale_x = orig_w / target_size
                scale_y = orig_h / target_size
                
                # --- 4. Process Top Predictions ---
                # Limit to top 100 to speed up COCO eval
                if len(scores) > 100:
                    indices = np.argsort(scores)[-100:]
                    boxes = boxes[indices]
                    scores = scores[indices]
                    labels = labels[indices]
                
                for box, score, label in zip(boxes, scores, labels):
                    if score > score_threshold:
                        # Box is [x1, y1, x2, y2] in 448x448 coord
                        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])
                        
                        # Rescale to Original Image
                        x1 *= scale_x
                        y1 *= scale_y
                        x2 *= scale_x
                        y2 *= scale_y
                        
                        w, h = x2 - x1, y2 - y1
                        if w <= 0 or h <= 0: continue

                        # Map Categories
                        cat_id = int(label)
                        if category_mapping:
                            cat_id = category_mapping.get(cat_id, cat_id)

                        all_predictions.append({
                            'image_id': int(img_id),
                            'category_id': cat_id,
                            'bbox': [x1, y1, w, h], # COCO expects XYWH
                            'score': float(score)
                        })
                
                # Collect Ground Truth for F1 calculation
                anns = self.coco_test.loadAnns(self.coco_test.getAnnIds(imgIds=img_id))
                for ann in anns:
                    x, y, w, h = ann['bbox']
                    all_ground_truth.append({
                        'image_id': int(img_id),
                        'category_id': ann['category_id'],
                        'bbox': [float(x), float(y), float(w), float(h)]
                    })
        
        # --- Metrics Calculation ---
        coco_metrics = self._compute_coco_metrics(all_predictions)
        f1_metrics = self._compute_f1_metrics(all_predictions, all_ground_truth, classnames, iou_threshold)
        
        # Stats
        avg_time = float(np.mean(inference_times)) if inference_times else 0
        fps = 1000.0 / avg_time if avg_time > 0 else 0
        
        inference_stats = {
            'avg_ms': avg_time,
            'fps': fps,
            'total_time_sec': float(sum(inference_times) / 1000) if inference_times else 0
        }
        
        self._log_test_results({'coco_metrics': coco_metrics, 'f1_metrics': f1_metrics, 'inference_stats': inference_stats}, classnames)
        return {'coco_metrics': coco_metrics, 'f1_metrics': f1_metrics}
    
    def _prepare_input(self, image_np, device, target_size):
        """Resizes image to target_size (distorted) and normalizes."""
        try: cv2.setNumThreads(0)
        except: pass

        MEAN = np.array([0.485, 0.456, 0.406])
        STD = np.array([0.229, 0.224, 0.225])
        
        try:
            image_resized = cv2.resize(image_np, (target_size, target_size))
        except Exception:
            # Fallback for PIL
            image_resized = np.array(Image.fromarray(image_np).resize((target_size, target_size)))

        # Normalize
        image_norm = image_resized.astype(np.float32) / 255.0
        image_norm = (image_norm - MEAN) / STD
        
        return torch.from_numpy(image_norm.transpose(2, 0, 1)).float().to(device)

    def _compute_coco_metrics(self, predictions):
        metrics = {k: 0.0 for k in ['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']}
        if not predictions:
            return metrics
            
        try:
            with redirect_stdout(io.StringIO()):
                coco_dt = self.coco_test.loadRes(predictions)
                coco_eval = COCOeval(self.coco_test, coco_dt, 'bbox')
                coco_eval.evaluate()
                coco_eval.accumulate()
                coco_eval.summarize()
            
            stats = coco_eval.stats
            metrics['mAP'] = float(stats[0])
            metrics['AP50'] = float(stats[1])
            metrics['AP75'] = float(stats[2])
            metrics['AP_small'] = float(stats[3])
            metrics['AP_medium'] = float(stats[4])
            metrics['AP_large'] = float(stats[5])
        except Exception as e:
            self.logger.error(f"TEST: COCO eval failed: {e}")
        return metrics
    
    @staticmethod
    def _compute_iou(box1_xyxy, box2_xyxy):
        # --- FIX: Convert inputs to numpy arrays safely ---
        box1 = np.array(box1_xyxy, dtype=np.float32)
        box2 = np.array(box2_xyxy, dtype=np.float32)
        
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        inter_area = max(0, x2 - x1) * max(0, y2 - y1)
        if inter_area == 0: return 0.0
        
        box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
        box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
        
        return inter_area / (box1_area + box2_area - inter_area + 1e-6)
    
    def _compute_f1_metrics(self, predictions, ground_truths, classnames, iou_threshold=0.5):
        # Group by class
        gt_by_class = defaultdict(list)
        pred_by_class = defaultdict(list)
        
        for gt in ground_truths: 
            gt_by_class[gt['category_id']].append(gt['bbox'])
        for pred in predictions: 
            pred_by_class[pred['category_id']].append({'bbox': pred['bbox'], 'score': pred['score']})
        
        class_metrics = {}
        for class_id in range(1, len(classnames) + 1):
            gt_boxes = gt_by_class.get(class_id, [])
            preds = sorted(pred_by_class.get(class_id, []), key=lambda x: x['score'], reverse=True)
            
            tp, fp = 0, 0
            matched_gt = set()
            
            for pred in preds:
                px, py, pw, ph = pred['bbox']
                pred_xyxy = [px, py, px + pw, py + ph]
                
                best_iou = 0.0
                best_gt_idx = -1
                
                for idx, (gx, gy, gw, gh) in enumerate(gt_boxes):
                    if idx in matched_gt: continue
                    iou = self._compute_iou(pred_xyxy, [gx, gy, gx + gw, gy + gh])
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = idx
                
                if best_iou >= iou_threshold:
                    tp += 1
                    matched_gt.add(best_gt_idx)
                else:
                    fp += 1
            
            fn = len(gt_boxes) - len(matched_gt)
            precision = tp / (tp + fp + 1e-6)
            recall = tp / (tp + fn + 1e-6)
            f1 = 2 * precision * recall / (precision + recall + 1e-6)
            
            cname = classnames[class_id - 1] if (class_id - 1) < len(classnames) else str(class_id)
            class_metrics[cname] = {'precision': precision, 'recall': recall, 'f1': f1}
            
        macro = {
            'precision': 0.0, 'recall': 0.0, 'f1': 0.0
        }
        if class_metrics:
            macro['precision'] = float(np.mean([m['precision'] for m in class_metrics.values()]))
            macro['recall'] = float(np.mean([m['recall'] for m in class_metrics.values()]))
            macro['f1'] = float(np.mean([m['f1'] for m in class_metrics.values()]))
            
        return {'per_class': class_metrics, 'macro': macro}
    
    def _log_test_results(self, test_results, classnames):
        coco = test_results['coco_metrics']
        f1m = test_results['f1_metrics']
        
        # Dùng PRINT để bắt buộc hiện ra màn hình
        print("\n" + "="*40)
        print(f"📊 FINAL TEST RESULTS")
        print("="*40)
        print(f"► mAP (IoU=0.5:0.95): {coco['mAP']:.4f}")
        print(f"► AP50 (IoU=0.50)   : {coco['AP50']:.4f}")
        print(f"► Inference Speed   : {test_results['inference_stats']['fps']:.1f} FPS")
        print("-" * 40)
        print("► DETAIL PER CLASS (F1-Score):")
        for c, m in f1m['per_class'].items():
            print(f"  • {c:15s}: F1={m['f1']:.4f} | Prec={m['precision']:.2f} | Rec={m['recall']:.2f}")
        print("="*40 + "\n")

logger.info("Test Evaluator loaded (Fixed Typo)")

In [ ]:
# ============================================================================
# CELL 11: EXPORT UTILITIES (FIXED TRACING & DEVICE MISMATCH)
# ============================================================================
import torch
import torch.nn as nn
from pathlib import Path
import logging

# --- 1. WRAPPER CLASS ---
class DetectorExportWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        predictions = self.model(x)
        pred = predictions[0]
        return pred['boxes'], pred['scores'], pred['labels']

# --- 2. EXPORTER CLASS ---
class ModelExporter:
    def __init__(self, output_dir, logger):
        self.output_dir = Path(output_dir)
        self.logger = logger
        self.export_dir = self.output_dir / 'exports'
        self.export_dir.mkdir(parents=True, exist_ok=True)
    
    def export_torchscript(self, model, input_size):
        try:
            savepath = self.export_dir / 'model_traced.pt'
            
            # 1. Move to CPU
            device_cpu = torch.device('cpu')
            model_cpu = model.to(device_cpu).eval()
            
            # 2. Wrap model
            wrapped_model = DetectorExportWrapper(model_cpu).eval()
            dummy_input = torch.randn(1, 3, input_size, input_size).to(device_cpu)
            
            # --- FIX: WARM-UP FORWARD PASS ---
            # Bắt buộc chạy 1 lần trên CPU để AnchorGenerator cập nhật device từ CUDA -> CPU
            with torch.no_grad():
                wrapped_model(dummy_input)
            
            # 3. Trace with strict checks disabled
            with torch.no_grad():
                # check_trace=False: Bỏ qua lỗi "Constant nodes differed" do Anchor sinh động
                # strict=False: Cho phép Dictionary output (nếu không dùng wrapper)
                traced_model = torch.jit.trace(wrapped_model, dummy_input, check_trace=False)
            
            traced_model.save(str(savepath))
            self.logger.info(f"EXPORT: TorchScript saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"EXPORT: TorchScript failed: {e}")
            return None
    
    def export_onnx(self, model, input_size, opset_version=12):
        try:
            import onnx
            output_path = self.export_dir / 'model.onnx'
            
            device_cpu = torch.device('cpu')
            model_cpu = model.to(device_cpu).eval()
            wrapped_model = DetectorExportWrapper(model_cpu).eval()
            dummy_input = torch.randn(1, 3, input_size, input_size).to(device_cpu)
            
            # Warm-up on CPU
            with torch.no_grad():
                wrapped_model(dummy_input)

            with torch.no_grad():
                torch.onnx.export(
                    wrapped_model, dummy_input, str(output_path),
                    opset_version=opset_version, do_constant_folding=True,
                    input_names=['input'], output_names=['boxes', 'scores', 'labels'],
                    dynamic_axes={'input': {0: 'batch'}, 'boxes': {0: 'num_dets'}, 'scores': {0: 'num_dets'}, 'labels': {0: 'num_dets'}},
                    verbose=False
                )
            onnx_model = onnx.load(str(output_path))
            onnx.checker.check_model(onnx_model)
            self.logger.info(f"EXPORT: ONNX saved and verified: {output_path}")
            return output_path
        except Exception as e:
            self.logger.error(f"EXPORT: ONNX failed: {e}")
        return None

    def export_checkpoint(self, model, optimizer, scheduler, epoch, metrics, config, filename=None):
        try:
            name = filename if filename else f'checkpoint_epoch_{epoch}.pth'
            savepath = self.export_dir / name
            
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
                'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
                'config': config,
                'metrics': metrics
            }
            torch.save(checkpoint, str(savepath))
            self.logger.info(f"EXPORT: Checkpoint saved: {savepath}")
            return savepath
        except Exception as e:
            self.logger.error(f"EXPORT: Checkpoint failed: {e}")
            return None

# --- MAIN EXPORT RUNNER ---
def run_exports(model, device, config, logger, optimizer=None, scheduler=None, epoch=None):
    if not config.get('EXPORT_MODEL', False): return {}
    logger.info("-" * 40); logger.info("EXPORT: Starting model exports...")
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    exporter = ModelExporter(config['OUTPUT_DIR'], logger)
    input_size = config.get('BASE_INPUT_SIZE', 448)
    exports = {}
    
    # Pass optimizer=None explicitly
    exports['weights'] = exporter.export_checkpoint(model, None, None, epoch or 0, {}, config, filename='model_weights_only.pth')
    exports['torchscript'] = exporter.export_torchscript(model, input_size)
    exports['onnx'] = exporter.export_onnx(model, input_size)

    if optimizer and epoch is not None:
        exports['checkpoint'] = exporter.export_checkpoint(model, optimizer, scheduler, epoch, {}, config)

    model.to(device)
    logger.info(f"EXPORT: Completed."); logger.info("-" * 40)
    return exports

logger.info("Export utilities loaded (Fixed Tracing & Device Error)")

In [ ]:
# ============================================================================
# CELL 11.5: LOG MERGING FUNCTION (Defined before Main)
# ============================================================================
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def merge_and_plot_history(config, logger):
    """
    Hàm này tự động tìm log cũ (từ input) và log mới (vừa train xong),
    nối chúng lại và vẽ biểu đồ toàn vẹn quá trình huấn luyện.
    """
    logger.info("Starting Log Merging for Publication Plots...")
    
    # 1. Cấu hình đường dẫn (CẦN KIỂM TRA LẠI PATH NÀY KHỚP VỚI DATASET CỦA BẠN)
    # Đây là đường dẫn đến log của 50 epoch đầu tiên (trong input dataset)
    path_stage_1 = Path('/kaggle/input/checkpoint-mobilenetv3-2/logs') 
    
    # Đường dẫn log hiện tại (trong working directory)
    path_stage_2 = config['OUTPUT_DIR'] / 'logs'
    
    def load_and_merge(filename):
        file_1 = path_stage_1 / filename
        file_2 = path_stage_2 / filename
        df_final = pd.DataFrame()
        
        # Load Stage 1
        if file_1.exists():
            try:
                df1 = pd.read_csv(file_1)
                logger.info(f"Loaded Stage 1 ({filename}): {len(df1)} rows")
                df_final = df1
            except: pass
        
        # Load Stage 2
        if file_2.exists():
            try:
                df2 = pd.read_csv(file_2)
                logger.info(f"Loaded Stage 2 ({filename}): {len(df2)} rows")
                if not df_final.empty:
                    last_epoch = df_final['epoch'].max()
                    df2['epoch'] = df2['epoch'] + last_epoch # Nối tiếp số epoch
                    df_final = pd.concat([df_final, df2], ignore_index=True)
                else:
                    df_final = df2
            except: pass
        return df_final

    # 2. Thực hiện ghép nối
    full_epoch_df = load_and_merge('epoch_metrics.csv')
    full_val_df = load_and_merge('validation_metrics.csv')
    
    if full_epoch_df.empty and full_val_df.empty:
        logger.warning("Không đủ dữ liệu để ghép nối log.")
        return

    # 3. Vẽ biểu đồ tổng hợp
    visualizer = ScientificVisualizer(config['OUTPUT_DIR'], logger)
    generated_files = visualizer.generate_all_figures(full_epoch_df, full_val_df)
    
    logger.info(f"MERGED PLOTS GENERATED: {len(generated_files)} files in plots/figures")

    # 4. Vẽ biểu đồ đặc biệt (mAP History Full)
    if 'mAP' in full_val_df.columns:
        try:
            plt.figure(figsize=(10, 6))
            plt.plot(full_val_df['epoch'], full_val_df['mAP'], marker='o', linewidth=2, color='#0072B2', label='Validation mAP')
            if path_stage_1.exists(): # Vẽ vạch ngăn cách giai đoạn
                try:
                    df1_temp = pd.read_csv(path_stage_1 / 'validation_metrics.csv')
                    plt.axvline(x=df1_temp['epoch'].max(), color='red', linestyle='--', alpha=0.7, label='Fine-tune Start')
                except: pass
            
            plt.title(f"Full Training History (Pretrain + Fine-tune)\nMax mAP: {full_val_df['mAP'].max():.4f}", fontsize=14)
            plt.xlabel("Epoch")
            plt.ylabel("mAP")
            plt.grid(True, alpha=0.3)
            plt.legend()
            save_path = config['OUTPUT_DIR'] / 'plots' / 'figures' / 'fig_special_full_history.png'
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
            logger.info(f"Saved special plot: {save_path}")
        except Exception as e:
            logger.warning(f"Failed to plot special history: {e}")

In [ ]:
# ============================================================================
# CELL 12: MAIN TRAINING PIPELINE (FINAL INTEGRATION)
# ============================================================================
import os

def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if len(batch) == 0: return None
    return [item[0] for item in batch], [item[1] for item in batch]

def main():
    # ... (Giữ nguyên nội dung hàm main như cũ) ...
    # Để tiết kiệm không gian, tôi không paste lại toàn bộ nội dung hàm main() 
    # vì bạn không thay đổi logic bên trong nó.
    # Hãy đảm bảo hàm main() ở đây giống hệt hàm main() ở bước trước.
    
    logger.info("=" * 80)
    logger.info("MAIN PIPELINE START (MobileNetV3 + EMA)")
    logger.info("=" * 80)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Setup Data
    train_dataset = MultiScaleTrafficDataset(CONFIG['TRAIN_IMG'], CONFIG['TRAIN_ANN'], CONFIG, train=True)
    val_dataset = MultiScaleTrafficDataset(CONFIG['VAL_IMG'], CONFIG['VAL_ANN'], CONFIG, train=False)
    model_label2coco_id = {v: k for k, v in train_dataset.catid2label.items()}

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, num_workers=CONFIG['NUM_WORKERS'], persistent_workers=True, collate_fn=collate_fn, pin_memory=CONFIG['PIN_MEMORY'])
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=CONFIG['NUM_WORKERS'], persistent_workers=True, collate_fn=collate_fn, pin_memory=CONFIG['PIN_MEMORY'])

    # 2. Setup Model & Optimizer
    model = build_model(CONFIG['NUM_MODEL_CLASSES'], device)
    
    # Load Pretrained for Fine-tuning
    if CONFIG.get('PRETRAINED_WEIGHTS') and os.path.exists(CONFIG['PRETRAINED_WEIGHTS']):
        try:
            checkpoint = torch.load(CONFIG['PRETRAINED_WEIGHTS'], map_location=device, weights_only=False)
            state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
            model.load_state_dict(state_dict)
            logger.info("Loaded pretrained weights for Fine-tuning.")
        except Exception as e: logger.warning(f"Load pretrained failed: {e}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LEARNING_RATE'], weight_decay=CONFIG['WEIGHT_DECAY'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['EPOCHS'], eta_min=1e-6)
    scaler = GradScaler() if CONFIG.get('USE_AMP', False) else None

    # 3. EMA
    ema = None
    if CONFIG.get('EMA_ENABLED', True):
        logger.info("Initializing ModelEMA...")
        ema = ModelEMA(model, decay=CONFIG.get('EMA_DECAY', 0.9999))

    # 4. Resume
    start_epoch = 0
    best_val_map = 0.0
    if CONFIG.get('RESUME_TRAINING') and os.path.exists(CONFIG['RESUME_CHECKPOINT']):
        try:
            checkpoint = torch.load(CONFIG['RESUME_CHECKPOINT'], map_location=device, weights_only=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if 'scheduler_state_dict' in checkpoint and scheduler: scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            start_epoch = checkpoint.get('epoch', 0) + 1
            best_val_map = checkpoint.get('best_val_map', 0.0)
            if ema and 'ema_state_dict' in checkpoint:
                ema.ema.load_state_dict(checkpoint['ema_state_dict'])
                ema.updates = checkpoint.get('ema_updates', 0)
            logger.info(f"Resumed from Epoch {start_epoch}.")
        except Exception as e: logger.warning(f"Resume failed: {e}")

    # 5. Loop
    checkpoint_path = os.path.join(CONFIG['CHECKPOINT_DIR'], 'latest.pth')
    best_model_path = os.path.join(CONFIG['CHECKPOINT_DIR'], 'best.pth')

    for epoch in range(start_epoch, CONFIG['EPOCHS']):
        metrics = train_one_epoch(model, optimizer, train_loader, device, epoch+1, scaler, scheduler, CONFIG, metrics_logger, ema=ema)
        
        if (epoch + 1) % CONFIG['VAL_FREQUENCY'] == 0:
            eval_model = ema.ema if ema else model
            val_res = validate_model(eval_model, val_loader, device, CONFIG, logger, category_id_map=model_label2coco_id)
            metrics_logger.log_validation(epoch+1, val_res, val_res['inference_time'])
            
            if val_res['mAP'] > best_val_map:
                best_val_map = val_res['mAP']
                torch.save(eval_model.state_dict(), best_model_path)
                logger.info(f"*** NEW BEST mAP: {best_val_map:.4f} ***")

        save_dict = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'best_val_map': best_val_map, 'config': CONFIG
        }
        if ema:
            save_dict['ema_state_dict'] = ema.ema.state_dict()
            save_dict['ema_updates'] = ema.updates
        torch.save(save_dict, checkpoint_path)

    return {'best_map': best_val_map, 'device': device}

if __name__ == '__main__':
    try:
        # 1. Run Training
        result = main()
        device = result['device']
        best_model_path = os.path.join(CONFIG['CHECKPOINT_DIR'], 'best.pth')
        
        # 2. Generate Plots (Ngay sau khi train xong)
        logger.info("Generating standard training plots...")
        generate_publication_figures(CONFIG, logger)
        
        # 3. Merge & Plot History (Vẽ biểu đồ ghép nối Pretrain + Fine-tune)
        # Hàm này được định nghĩa ở Cell 11.5
        merge_and_plot_history(CONFIG, logger)

        # 4. Post-Training Operations
        if os.path.exists(best_model_path):
            logger.info("Loading best model for Export & Test Evaluation...")
            
            final_model = build_model(CONFIG['NUM_MODEL_CLASSES'], device)
            # Load weights safely
            final_model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=False))
            final_model.to(device)
            final_model.eval()

            # Test Evaluation
            if CONFIG.get('RUN_TEST_EVAL', False):
                evaluator = ComprehensiveEvaluator(
                    test_img_dir=CONFIG['TEST_IMG'], 
                    test_ann_file=CONFIG['TEST_ANN'],
                    logger=logger, config=CONFIG
                )
                evaluator.evaluate_on_test_set(
                    model=final_model, 
                    device=device, 
                    classnames=USER_CLASSES_LIST,
                    score_threshold=0.05
                )
            
            # Export
            if CONFIG.get('EXPORT_MODEL', False):
                run_exports(final_model, device, CONFIG, logger)
                
    except Exception as e:
        logger.error(f"CRITICAL PIPELINE ERROR: {e}")
        import traceback
        traceback.print_exc()